# S6E11 | 44-Model Pooled Stack: Digit-Trie + Seed-Bagged TE + Public OOF Banks | LB 0.95180 (fusion) · 0.95170 (validated)

**TL;DR**
* The competition data is an **LLM-written copy** of a small dataset that was itself generated by a near-additive logistic formula. The copy leaves **label signal inside the digits of the numbers**.
* Target-encoding every number at **several resolutions** (exact value, cents, `int(10x)` / `int(x)` / `int(x/10)`, leading digits) and by its **GPT-2 tokens** gets the best single model to ~0.9504 CV AUC.
* **Seed bagging of the target encoding** (averaging over inner-split seeds) lifts the best single model **0.95037 → 0.95077**, and a **full-data refit** of the bagged members adds a little more on the leaderboard.
* **New in this notebook: pooling public out-of-fold banks.** Nilanjan_Saha123's 96-key models and goodpjw2008's updated artifacts publish OOF + refit predictions. Adding them to our pool lifts nested CV **0.95103 → 0.95110** and the public LB **0.95145 → 0.95170**.
* **Anchor fusion (leaderboard only):** rank-averaging the public 0.95180 submission 85/15 with our validated blend scored **0.95180** (50/50 gave 0.95178, 70/30 gave 0.95179). It is written to a separate file and explained in section 5, because CV cannot check it. `submission.csv` is the validated blend.

This notebook is the **final blending stage**: every member was trained elsewhere (section 2) and saved as out-of-fold (OOF) and test predictions. Everything printed below is real output of this run.

## 1. What we learned — every claim measured fold by fold
All numbers are 5-fold CV AUC on the 700k training rows. A feature group was kept only if it improved the mean **and** won on at least 4 of 5 folds.

| Experiment | Effect on CV AUC | Folds better | Verdict |
|---|---|---|---|
| Plain XGBoost on raw columns | 0.94703 | – | baseline |
| Per-value target encoding of every column | **+0.00110** | 5/5 | kept |
| Raw digit features (decimal count, last digit) | +0.00019 | 5/5 | superseded |
| **Multi-resolution "digit-trie" encodings** (value, cents, rounded resolutions, leading digits) | best single **0.9488 → 0.9504** | – | **kept — the big win** |
| GPT-2 token keys (numbers padded to fixed decimals) | +0.00012 | 3/5 | diversity members only |
| GPT-2 token keys (numbers exactly as written) | +0.00012 | 3/5 | diversity members only |
| Target encoding of 45 column pairs | −0.00070 | 1/5 | rejected |
| Nearest-neighbour "near-duplicate" label features | −0.00029 | 0/5 | rejected |
| Count encoding / value-in-original flags / recovered original formula / flag count | ≈0 | ≤2/5 | rejected |
| Target-encoding smoothing 5 → 400 | flat (±0.0001) | – | default 20 kept |
| Optuna-tuned depth-2 XGBoost vs default | +0.00016 | 4/5 | kept |
| RealMLP on the encodings | 0.94924 OOF (best neural net) | – | diversity member |
| Trie-smoothed target encoding (rare values shrunk toward their parent bucket, not the global rate) | −0.00010 | 1/5 | rejected |
| Row-level formatting counts (decimals, trailing zeros, round numbers across the row) | −0.00012 | 0/5 | rejected |
| Target encoding of row-level formatting patterns | −0.00007 | 2/5 | rejected |
| Row order (`id`): fraud rate of neighbouring ids as a predictor | AUC 0.496–0.500 | – | no leak |
| **Seed bagging of the target encoding** (same model, 5 inner-split seeds, averaged) | best single **0.95037 → 0.95074** | – | **kept: second big win** |
| Full-data refit for the bagged members' test predictions | +0.00007 public LB | – | kept (CV can't measure it) |
| 13 encoding seeds instead of 8 | CV +0.00001, LB −0.00005 | – | saturated |
| **Pooling public OOF banks** (96-key GLM / LightGBM / XGBoost, updated goodpjw2008 artifacts) | nested CV **0.95103 → 0.95110**, LB **0.95145 → 0.95170** | – | **kept: third big win** |

**Why digits matter.** The original 20k-row dataset is close to a logistic formula (a logistic regression fitted on it scores ~0.935 on the competition data untouched). The competition copy shows the fingerprints of LLM generation (low-temperature sampling, tokenizer artefacts — see Hemingweb's EDA in the credits). An LLM writes a number piece by piece, so the integer part, the cents and the leading digits each carry label information of their own. Encoding each piece separately captures that; encoding only the full value does not.

**Why bagging the encoding works.** Each training row's target encoding is computed out of fold from an *inner* 5-fold split. With rare values (most exact amounts appear only a few times), a different inner split gives a noticeably different encoding, and the model fits that noise. Re-running the whole pipeline with several inner-split seeds and averaging the predictions cancels most of it. The table below shows single-model OOF AUC for the 4-leaf XGBoost on identical features:

| Encoding seeds averaged | OOF AUC |
|---|---|
| 1 | 0.95037 |
| 3 | 0.95061 |
| 5 | 0.95074 |
| 5 (depth-2 XGBoost) | 0.95077 |

**Full-data refit.** For the bagged members, the training-set encodings are rebuilt out of fold over all 700k rows, the test encodings are fitted on all 700k rows, and the model is refitted with 1.15 × the mean early-stopping iteration from the folds. The refit test predictions correlate 0.996–0.998 with the 5-fold average: close, but on the public LB the refit version won every time we compared (0.95135 → 0.95142 on the same blend).

**Honest negatives.** Near-duplicate clusters (a natural guess for low-temperature sampling) did *not* help, and neither did pairwise encodings — both are kept in the table because knowing what fails saves everyone time.

## 2. Where the members come from
Every member is an out-of-fold prediction on the same 700k training rows plus a matching test prediction; predictions are converted to ranks so they share one scale.

| Source run | Prefix in member names | Members | Key features |
|---|---|---|---|
| Digit-trie run | `notebooka23a:` | `xgb_l4`, `lgb_l4`, `xgb_l4_s100`, `lgb_l4_s100`, `xgb_d2`, `glm`, `cat_d4`, `mlp`, `xgb_raw`, `lgb_raw` | value / cents / rounded / leading-digit encodings; Optuna-tuned 4-leaf trees; smoothing 20 and 100 views |
| Run v6 | `notebook9f8b:` | `xgb_l4`, `xgb_d2`, `glm_te` | same encodings, new seeds; logistic regression on logit(encodings) |
| Run v7 | `notebook477e:` | `xgb_l4`, `xgb_l4_tok`, `glm_tok` | + GPT-2 token keys (raw and padded) |
| Run v8 | `notebookf04b:` | `xgb_l4_knn`, `glm_knn` | + nearest-neighbour features (rejected as a feature, kept only if the blend wants them) |
| Run v9 | `notebook4b94:` | `realmlp_tok` | RealMLP (pytabkit) on encodings + token keys |
| Run v10 | `s6e11-v10-tr:` | `xgb_l4_v10`, `xgb_l4_v10_s7`, `glm_v10`, `realmlp_v10` | + the rejected trie-smoothed and row-format encodings (diversity only) |
| Run v11 | `s6e11-v11-ba:` | `xgb_bag`, `glm_bag` | 4-leaf XGBoost / GLM averaged over **3 encoding seeds**, plus full-data refit |
| Run v12 | `s6e11-v12-ba:` | `xgb_bag5`, `xgbd2_bag5`, `glm_bag5` | 4-leaf and depth-2 XGBoost / GLM averaged over **5 more encoding seeds**, plus full-data refit |
| Run v13 | `s6e11-v13-ba:` | `xgb_bag5b`, `xgbd2_bag5b`, `glm_bag5b` | 5 further encoding seeds, plus full-data refit |
| Nilanjan_Saha123's public **96-Key Digit-Trie + GPT-2** notebook | `96-key-digit:` | `glm_unified_96key`, `lgb_unified_96key_bag`, `xgb_unified_96key_bag` | joint sparse GLM and bagged shallow trees on 96 keys (24 exact, 36 digit-trie, 36 GPT-2 token), full refit |
| goodpjw2008's **S6E11 Fraud Series Artifacts** (public dataset) | `s6e11-fraud-:` | `ebm_b4`, `ebm_dig`, `realmlp_*`, plus the newer `glm_ohk_c01`, `lgb_dig_l7`, `lgb_dig_s100_bag`, `lgb_diglead_bag`, `lgb_rawdig_l4` | explainable boosting machines, RealMLPs, and (newer version) digit-key LightGBMs and a one-hot GLM with refit predictions |

Members more than 0.006 AUC below the best are dropped before blending (they rarely help and slow the search).

## 3. Load and check every member

In [ ]:
import os, glob, time, json, warnings
import numpy as np, pandas as pd
from scipy.stats import rankdata
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LogisticRegression
warnings.filterwarnings("ignore")
try:
    display
except NameError:
    display = print
T0 = time.time()
def log(*a): print(f"[{time.time()-T0:7.1f}s]", *a, flush=True)
ROOT = os.environ.get("KAGGLE_INPUT", "/kaggle/input"); OUT = os.environ.get("KAGGLE_OUT", "/kaggle/working")
files = sorted(glob.glob(f"{ROOT}/**/*.csv", recursive=True))
comp = [f for f in files if "/competitions/" in f] or files
TRAIN = [f for f in comp if os.path.basename(f) == "train.csv"][0]
SUB = [f for f in comp if os.path.basename(f) == "sample_submission.csv"][0]
log("train:", TRAIN)
tr = pd.read_csv(TRAIN, usecols=["id", "is_fraud"], dtype=str, keep_default_na=False)
BOOL_MAP = {"true": 1, "false": 0, "1": 1, "0": 0, "1.0": 1, "0.0": 0, "yes": 1, "no": 0}
y = tr["is_fraud"].str.strip().str.lower().map(BOOL_MAP)
log("label values:", tr["is_fraud"].str.strip().str.lower().value_counts().head(5).to_dict())
y = y.astype(int).values
NTR = len(y); sub = pd.read_csv(SUB); NTE = len(sub)
FOLDS = list(StratifiedKFold(5, shuffle=True, random_state=42).split(np.zeros(NTR), y))

In [ ]:
paths = {}; FULL = {}
GENERIC = {"oof", "preds", "artifacts", "output", "outputs", "working"}
def owner_of(dn):
    b = os.path.basename(dn)
    return os.path.basename(os.path.dirname(dn)) if b in GENERIC else b
for f in sorted(glob.glob(f"{ROOT}/**/*.npy", recursive=True)):
    bn, dn = os.path.basename(f), os.path.dirname(f)
    if bn.startswith("oof_"): base, t = bn[4:-4], os.path.join(dn, "test_" + bn[4:])
    elif bn.endswith("_oof.npy"):
        base = bn[:-8]; t = os.path.join(dn, base + "_test.npy")
        if not os.path.exists(t): t = os.path.join(dn, base + "_full.npy")   # some public runs only save refit test preds
    else: continue
    n = owner_of(dn)[:12] + ":" + base
    if os.path.exists(t) and n not in paths:
        o_, t_ = np.load(f), np.load(t)
        if len(o_) == NTR and len(t_) == NTE:
            paths[n] = (o_, t_)
            fl = os.path.join(dn, "full_" + base + ".npy")
            if os.path.exists(fl): FULL[n] = np.load(fl)
            if t.endswith("_full.npy"): FULL[n] = t_
        else: log("skip (shape)", n, o_.shape, t_.shape)
# .npz banks: pair every array of length NTR whose key mentions oof with a test/full array of length NTE
for f in sorted(glob.glob(f"{ROOT}/**/*.npz", recursive=True)):
    try:
        z = np.load(f, allow_pickle=False); keys = list(z.keys())
        log("npz", os.path.basename(f), {k: z[k].shape for k in keys})
        for k in keys:
            if "oof" not in k.lower() or z[k].ndim != 1 or len(z[k]) != NTR: continue
            for alt in (k.replace("oof", "test"), k.replace("oof", "full"), k.replace("oof", "tst"), k.replace("OOF", "TEST")):
                if alt in z and z[alt].ndim == 1 and len(z[alt]) == NTE:
                    n = owner_of(os.path.dirname(f))[:12] + ":" + k.replace("_oof", "").replace("oof_", "")
                    paths[n] = (z[k], z[alt]); log("  npz member", n, "<-", k, "+", alt); break
    except Exception as e:
        log("npz skip", f, str(e)[:120])
names = list(paths)
O = np.column_stack([rankdata(paths[n][0]) / NTR for n in names])
Tm = np.column_stack([rankdata(paths[n][1]) / NTE for n in names])
auc = {n: float(roc_auc_score(y, O[:, i])) for i, n in enumerate(names)}
log("members with full-data refit test predictions:", list(FULL))
log(f"{len(names)} members"); print(pd.Series(auc).sort_values(ascending=False).round(5).to_string(), flush=True)
# drop weak members to keep the search sane (they rarely help and slow hill climbing)
thr = max(auc.values()) - 0.006
keep = [i for i, n in enumerate(names) if auc[n] >= thr]
names = [names[i] for i in keep]; O = O[:, keep]; Tm = Tm[:, keep]
log(f"{len(names)} members kept (OOF AUC >= {thr:.5f})")

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
s_auc = pd.Series({n: auc[n] for n in names}).sort_values()
fig, ax = plt.subplots(figsize=(9, 0.32 * len(s_auc) + 1))
ax.barh(s_auc.index, s_auc.values, color="#2a6f97"); ax.set_xlim(s_auc.min() - 0.001, s_auc.max() + 0.0006)
ax.set_xlabel("out-of-fold AUC"); ax.set_title("Members used in the blend"); plt.tight_layout(); plt.show()
corr = pd.DataFrame(O, columns=names).corr()
fig, ax = plt.subplots(figsize=(10, 8)); im = ax.imshow(corr.values, cmap="viridis")
ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=90, fontsize=7)
ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=7)
plt.colorbar(im, ax=ax); ax.set_title("Rank correlation between members (lower = more diverse)"); plt.tight_layout(); plt.show()

## 4. Blend, chosen by nested cross-validation
Five candidates are compared **out of fold**: blend weights are fitted on 4 folds of the OOF matrix and scored on the 5th, so the comparison is not flattered by fitting and scoring on the same rows.
* `mean` — rank average of all members
* `hill` — greedy hill climbing with replacement (Caruana-style ensemble selection)
* `stack` / `stack_c01` — logistic regression on the members' logits (two regularisation strengths)
* `mix` — average of the hill-climb and stack ranks

The winner is refitted on all OOF rows and applied to the test predictions. Output files: `submission.csv` is this blend with full-data refit test predictions for every member that has them (**0.95170**, the CV-validated result). `submission_foldavg.csv` uses 5-fold-average test predictions instead. `submission_fusion.csv` is the anchor fusion from section 5 (**0.95180**).

In [ ]:
def hill_climb(O, yy, iters=80):
    w = np.zeros(O.shape[1]); w[int(np.argmax([roc_auc_score(yy, O[:, i]) for i in range(O.shape[1])]))] = 1
    cur = roc_auc_score(yy, O @ w)
    for _ in range(iters):
        sc, i = max((roc_auc_score(yy, O @ (w + np.eye(O.shape[1])[i])), i) for i in range(O.shape[1]))
        if sc <= cur + 1e-7: break
        w[i] += 1; cur = sc
    return w / w.sum()
def logit(p): p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))
cv = {k: np.zeros(NTR) for k in ["mean", "hill", "stack", "stack_c01", "mix"]}
for a, b in FOLDS:
    cv["mean"][b] = O[b].mean(1)
    cv["hill"][b] = O[b] @ hill_climb(O[a], y[a])
    cv["stack"][b] = LogisticRegression(C=1.0, max_iter=3000).fit(logit(O[a]), y[a]).decision_function(logit(O[b]))
    cv["stack_c01"][b] = LogisticRegression(C=0.01, max_iter=3000).fit(logit(O[a]), y[a]).decision_function(logit(O[b]))
    cv["mix"][b] = rankdata(cv["hill"][b]) / len(b) + rankdata(cv["stack"][b]) / len(b)
scores = {k: float(roc_auc_score(y, v)) for k, v in cv.items()}
log("nested-CV blend AUC:", {k: round(float(v), 6) for k, v in scores.items()})
best = max(scores, key=scores.get)
w = hill_climb(O, y)
st = LogisticRegression(C=1.0, max_iter=3000).fit(logit(O), y)
st01 = LogisticRegression(C=0.01, max_iter=3000).fit(logit(O), y)
final = {"mean": Tm.mean(1), "hill": Tm @ w, "stack": st.decision_function(logit(Tm)),
         "stack_c01": st01.decision_function(logit(Tm))}
final["mix"] = rankdata(final["hill"]) / NTE + rankdata(final["stack"]) / NTE
log("hill weights:", {n: round(float(v), 3) for n, v in zip(names, w) if v > 0})
sub_fold = sub.copy(); sub_fold["is_fraud"] = rankdata(final[best]) / NTE
sub_fold.to_csv(f"{OUT}/submission_foldavg.csv", index=False)
# same weights, but the bagged members' test columns come from the full-data refit
Tm2 = Tm.copy()
for j, n in enumerate(names):
    if n in FULL: Tm2[:, j] = rankdata(FULL[n]) / NTE
f2 = {"mean": Tm2.mean(1), "hill": Tm2 @ w, "stack": st.decision_function(logit(Tm2)), "stack_c01": st01.decision_function(logit(Tm2))}
f2["mix"] = rankdata(f2["hill"]) / NTE + rankdata(f2["stack"]) / NTE
sub["is_fraud"] = rankdata(f2[best]) / NTE
sub.to_csv(f"{OUT}/submission.csv", index=False)   # the CV-validated blend
log("refit swapped for", [n for n in names if n in FULL], "| rank corr refit vs fold-avg:",
    round(float(np.corrcoef(sub["is_fraud"], sub_fold["is_fraud"])[0, 1]), 5))
json.dump(dict(members=auc, kept=names, blend=scores, best=best), open(f"{OUT}/summary.json", "w"), indent=1)
log(f"BEST BLEND = {best} | nested CV {scores[best]:.6f} | rows {len(sub)}")

In [ ]:
res = pd.Series(scores).sort_values(ascending=False).round(6)
display(res.to_frame("nested-CV AUC"))
wd = pd.Series({n: v for n, v in zip(names, w) if v > 0}).sort_values()
fig, ax = plt.subplots(figsize=(8, 0.35 * len(wd) + 1)); ax.barh(wd.index, wd.values, color="#d1495b")
ax.set_xlabel("weight"); ax.set_title("Hill-climbing weights (part of the final 'mix')"); plt.tight_layout(); plt.show()

## 5. Optional: anchor fusion (leaderboard only, read this before copying it)
The public **96-Key Digit-Trie + GPT-2** notebook's own submission scores 0.95180. Its final step blends in predictions that have no OOF counterpart, so it cannot be scored out of fold. Here it is rank-averaged with our validated blend. These are the weights we actually submitted:

| Anchor weight | Our weight | Public LB |
|---|---|---|
| 0 | 1 | 0.95170 (validated blend) |
| 0.5 | 0.5 | 0.95178 |
| 0.7 | 0.3 | 0.95179 |
| **0.85** | **0.15** | **0.95180** (this file) |
| 1 | 0 | 0.95180 (anchor alone) |

A quadratic through the first, second and last points predicted the peak near 0.85–0.9, at about the anchor's own score. The submissions matched that, so mixing these two sources alone cannot get meaningfully above 0.95180.

* **Why it can help:** the two pipelines share ideas but differ in members and blending, so their errors are not identical (rank correlation printed below).
* **Why to be careful:** the weight cannot be tuned by CV. The only evidence is the public leaderboard, which covers ~20% of the test rows and is noisy at the 1e-4 level. Treat it as one of your two final picks, not both.

In [ ]:
anc = [f for f in files if os.path.basename(f) == "submission.csv" and "96-key" in f.lower()]
if anc:
    a_rank = rankdata(pd.read_csv(anc[0]).set_index("id").loc[sub["id"], "is_fraud"].values) / NTE
    ours = sub["is_fraud"].values.copy()
    log("anchor:", anc[0], "| rank corr with our validated blend:", round(float(np.corrcoef(a_rank, ours)[0, 1]), 5))
    fused = sub.copy(); fused["is_fraud"] = rankdata(0.15 * ours + 0.85 * a_rank) / NTE
    fused.to_csv(f"{OUT}/submission_fusion.csv", index=False)
    log("submission_fusion.csv = 85/15 anchor fusion (public LB 0.95180); submission.csv = CV-validated blend (0.95170)")
else:
    log("anchor not attached: no fusion file written")

## 6. Results, reproducibility and credits
**Result:** nested-CV AUC **0.95110** and **0.95170** public LB for the validated blend (`submission.csv`). The anchor fusion (`submission_fusion.csv`) scored **0.95180**.

**How the score moved:** every row below is a real submission.

| Version | What changed | Nested CV | Public LB |
|---|---|---|---|
| Previous version of this notebook | 24 members: digit-trie + token encodings, goodpjw2008 artifacts | 0.95097 | 0.95125 |
| + v10 members | rejected encodings as diversity members | 0.95096 | 0.95124 |
| + 3-seed bagging | `xgb_bag`, `glm_bag` | 0.95095 | 0.95129 |
| + 5-seed bagging | `xgb_bag5`, `xgbd2_bag5`, `glm_bag5` | 0.95101 | 0.95135 |
| + full-data refit | same blend, refit test predictions | 0.95101 | 0.95142 |
| Previous public version | all 33 members pooled, refit test predictions | 0.95103 | 0.95145 |
| (13 seeds) | 5 further seeds added | 0.95103 | 0.95140 |
| **+ public OOF banks** | 44 members: + 96-key models, + updated goodpjw2008 artifacts | **0.95110** | **0.95170** |
| anchor fusion 50/50 | with the public 0.95180 submission | – | 0.95178 |
| **anchor fusion 85/15** | with the public 0.95180 submission | – | **0.95180** |

More encoding seeds stopped helping after 8 (CV +0.00001, LB within noise). New, *different* members helped much more.

**Reproducibility.** The member runs are separate notebooks and datasets attached as inputs; their names appear as prefixes in the member list. Each one trains its members with 5-fold out-of-fold predictions, and this notebook only blends. If an input notebook publishes a new version, its predictions here change too.

**Lessons**
* The biggest gain came from understanding *how the data was made* (an LLM writing numbers piece by piece), not from tuning.
* Trust fold-by-fold CV: with ~1,000 frauds in a 20% public split, public-LB noise is roughly ±0.003 AUC, far larger than most gains here.
* Diverse members (neural nets, EBMs, logistic models on encodings) earn blend weight even when their single scores are lower.
* **If your features are target encodings, bag over the encoding seed.** It is the cheapest real gain we found after the digit-trie keys, and the blend gives the bagged model the largest weight (40%).
* **Refit the final members on all rows.** CV cannot reward it, but it helped on the leaderboard every time we compared.
* **Public OOF banks are worth more than more seeds.** Three well-built outside members gained more CV than five extra seeds of our own. Publishing OOF + test predictions, as Nilanjan_Saha123 and goodpjw2008 do, helps everyone.

### Credits and sources — please upvote their work too
* **Udit Jain** — the original [Credit Card Fraud Detection 2026](https://www.kaggle.com/datasets/uditjain13/credit-card-fraud-detection-2026) dataset (CC0).
* **Nilanjan_Saha123** — [96-Key Digit-Trie + GPT-2 | Full-Refit+Hybrid](https://www.kaggle.com/code/nilanjansaha123/96-key-digit-trie-gpt-2-full-refit-hybrid): its published OOF + full-refit predictions (96-key GLM, LightGBM, XGBoost) are blended here as members, and its submission is the anchor in section 5.
* **goodpjw2008** — [S6E11 Part 4 | Refit, Bag, Joint GLM](https://www.kaggle.com/code/goodpjw2008/s6e11-part-4-refit-bag-joint-glm-lb-0-95154) and the updated artifacts dataset (new digit-key LightGBM / GLM members with refit predictions).
* **goodpjw2008** — the [S6E11 notebook series](https://www.kaggle.com/code/goodpjw2008/s6e11-part-3-digits-leak-the-label-lb-0-95113) ("Digits Leak the Label"), which first showed per-value encoding with very shallow trees and digit-level signal, and the public [S6E11 Fraud Series Artifacts](https://www.kaggle.com/datasets/goodpjw2008/s6e11-fraud-series-artifacts) dataset, whose EBM and RealMLP predictions are blended here as members.
* **Hemingweb** — [S06E11 Deep EDA and list of all findings](https://www.kaggle.com/competitions/playground-series-s6e11/discussion/754727): evidence of an LLM-style copy (shared low sampling temperature, GPT-2 tokenizer fingerprint), which motivated the token experiments, and the public-LB noise estimate.
* **bulsahkecici** — [S6E11 | Value TE + Digit & Token Keys | LR Stack](https://www.kaggle.com/code/bulsahkecici/s6e11-value-te-digit-token-keys-lr-stack), which independently uses GPT-2 token keys.
* **ektarr** — [LB 0.95121 | RealMLP Stacking](https://www.kaggle.com/code/ektarr/lb-0-95121-realmlp-stacking), showing RealMLP's value in this competition's stacks.
* **Ensemble selection:** Caruana, Niculescu-Mizil, Crew, Ksikes (2004), *Ensemble Selection from Libraries of Models* (the hill-climbing blend used here).
* **RealMLP / pytabkit** — Holzmüller, Grinsztajn, Steinwart (2024), *Better by Default: Strong Pre-Tuned MLPs and Boosted Trees on Tabular Data*.
* **Kaggle** — Yao Yan, Walter Reade, Elizabeth Park. *Predicting Credit Card Fraud.* Playground Series S6E11, 2026.